# CrimsonVC Studio - Google Colab

A reproducible, Colab-first launcher for the compatible `src/ultimate_rvc` application in this repository.

### What this notebook provides

- A managed Python 3.12 environment instead of Colab's changing system Python.
- The CUDA dependency set declared by the project.
- Optional Google Drive persistence for models, datasets, output, and configuration.
- PyTorch, CUDA, GPU, BF16, Gradio, and disk-space verification before launch.
- Password protection for the temporary public Gradio link without saving the password.

### Quick start (RVC only)

1. In Colab, select **Runtime > Change runtime type > T4 GPU**.
2. Keep `google_drive` for persistent models, or select `runtime_only` for a quick test.
3. Select **Runtime > Run all** and approve Google Drive access when requested.
4. In cell 6, type a Web UI password of at least 8 characters and press Enter.
5. Open the `gradio.live` link and sign in as `crimsonvc` with that password.

Inside Gradio, upload the RVC `.pth` and optional `.index` under **Models & Train > Upload > Voice models**. For audio conversion use **Create > Song cover > One-click**; for text-to-speech conversion use **Create > Speech > One-click**.

> Use only audio and voices you own or have permission to process. A Gradio share URL is public-facing; keep authentication enabled and do not share the URL or password unnecessarily. Review the current Google Colab policies before use.

In [ ]:
# @title 1. Check the runtime and prepare the repository
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

repository_url = "https://github.com/DDME36/CrimsonVC-Studio.git"  # @param {type:"string"}
repository_revision = "main"  # @param {type:"string"}
reset_workspace = False  # @param {type:"boolean"}
require_gpu = True  # @param {type:"boolean"}

if not repository_url.startswith("https://github.com/"):
    raise ValueError(
        "repository_url must be a GitHub HTTPS repository URL."
    )

WORKSPACE = Path("/content/CrimsonVC")


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    """Run a command and stop immediately when it fails."""
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)


if require_gpu and shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU is attached. In Colab select Runtime > Change runtime type > GPU, then retry."
    )

if shutil.which("nvidia-smi"):
    run([
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ])

if reset_workspace and WORKSPACE.exists():
    if WORKSPACE.resolve() != Path("/content/CrimsonVC"):
        raise RuntimeError(f"Refusing to remove unexpected path: {WORKSPACE}")
    print(f"Removing the old ephemeral workspace: {WORKSPACE}")
    shutil.rmtree(WORKSPACE)

if not WORKSPACE.exists():
    WORKSPACE.mkdir(parents=True)
    run(["git", "init"], cwd=WORKSPACE)
    run(["git", "remote", "add", "origin", repository_url], cwd=WORKSPACE)
    run(["git", "fetch", "--depth", "1", "origin", repository_revision], cwd=WORKSPACE)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=WORKSPACE)
elif not (WORKSPACE / "pyproject.toml").is_file():
    raise RuntimeError(
        f"{WORKSPACE} exists but is not a valid CrimsonVC Studio checkout. Enable reset_workspace and retry."
    )
else:
    print(f"Reusing {WORKSPACE}. Enable reset_workspace to fetch a clean revision.")

os.chdir(WORKSPACE)
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=WORKSPACE, text=True
).strip()
print(f"Repository ready at commit {commit}: {WORKSPACE}")


In [ ]:
# @title 2. Configure persistent storage
storage_mode = "google_drive"  # @param ["google_drive", "runtime_only"]
drive_folder = "CrimsonVC"  # @param {type:"string"}

if storage_mode == "google_drive":
    from google.colab import drive

    drive.mount("/content/drive")
    data_root = Path("/content/drive/MyDrive") / drive_folder
    print("Google Drive persistence is enabled. Model training can be slower because Drive I/O is remote.")
else:
    data_root = Path("/content/CrimsonVC-data")
    print("Runtime-only storage is enabled. All generated data disappears when the runtime is deleted.")

models_dir = data_root / "models"
audio_dir = data_root / "audio"
config_dir = data_root / "config"
temp_dir = Path("/content/CrimsonVC-temp")
for directory in (models_dir, audio_dir, config_dir, temp_dir):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "URVC_MODELS_DIR": str(models_dir),
    "URVC_AUDIO_DIR": str(audio_dir),
    "URVC_CONFIG_DIR": str(config_dir),
    "URVC_TEMP_DIR": str(temp_dir),
    "URVC_CONSOLE_LOG_LEVEL": "WARNING",
    "GRADIO_ANALYTICS_ENABLED": "False",
    "PYTHONIOENCODING": "utf-8",
    "PYTHONUTF8": "1",
    "UV_CACHE_DIR": "/content/uv-cache",
    "UV_PROJECT_ENVIRONMENT": str(WORKSPACE / ".venv"),
})

print(f"Persistent data root: {data_root}")
print(f"Temporary files:      {temp_dir}")


In [ ]:
# @title 3. Install CrimsonVC Studio
python_version = "3.12"  # @param ["3.12"]
uv_version = "0.9.11"  # @param {type:"string"}
download_all_embedders = False  # @param {type:"boolean"}

run([sys.executable, "-m", "pip", "install", "--quiet", f"uv=={uv_version}"])
uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("uv was installed but its executable is not available on PATH.")

run([uv, "python", "install", python_version])
sync_command = [
    uv,
    "sync",
    "--python",
    python_version,
    "--extra",
    "cuda",
    "--no-dev",
    "--no-editable",
]
if (WORKSPACE / "uv.lock").is_file():
    sync_command.append("--locked")
run(sync_command, cwd=WORKSPACE)

# ContentVec is installed first; optional HuBERT/Spin models download on demand.
# Enable this only when you want to prefetch every bundled embedder.
os.environ["URVC_DOWNLOAD_ALL_EMBEDDERS"] = "1" if download_all_embedders else "0"
run(
    [uv, "run", "--no-sync", "python", "-m", "ultimate_rvc.core.main"],
    cwd=WORKSPACE,
    env=os.environ.copy(),
)
print("Installation and core initialization completed.")


In [ ]:
# @title 4. Verify the installed environment
verification_code = r'''
import json
import platform

import gradio
import torch

report = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    "gradio": gradio.__version__,
}
print(json.dumps(report, indent=2))
if not torch.cuda.is_available():
    raise SystemExit("The project environment cannot access CUDA. Stop here and reset the Colab runtime.")
'''
run(
    [uv, "run", "--no-sync", "python", "-c", verification_code],
    cwd=WORKSPACE,
    env=os.environ.copy(),
)

disk = shutil.disk_usage("/content")
print(f"Free runtime disk: {disk.free / 1024**3:.1f} GiB")
print("Environment verification passed.")


In [ ]:
# @title 5. Optional model compatibility benchmark
run_model_benchmark = False  # @param {type:"boolean"}
benchmark_suite = "inventory"  # @param ["inventory", "f0", "embedders", "all"]
include_language_embedders = False  # @param {type:"boolean"}

if run_model_benchmark:
    benchmark_command = [
        uv,
        "run",
        "--no-sync",
        "python",
        "scripts/benchmark_models.py",
        "--suite",
        benchmark_suite,
    ]
    if include_language_embedders:
        benchmark_command.append("--include-language-embedders")
    run(benchmark_command, cwd=WORKSPACE, env=os.environ.copy())
else:
    print("Benchmark skipped. Enable run_model_benchmark to test installed model backends.")


In [ ]:
# @title 6. Launch the password-protected Web UI
from getpass import getpass

enable_authentication = True  # @param {type:"boolean"}
auth_username = "crimsonvc"  # @param {type:"string"}

launch_env = os.environ.copy()
if enable_authentication:
    if not auth_username.strip():
        raise ValueError("auth_username cannot be empty when authentication is enabled.")
    auth_password = getpass("Create a Web UI password (minimum 8 characters): ")
    if len(auth_password) < 8:
        raise ValueError("Use a password with at least 8 characters.")
    launch_env["URVC_AUTH_USERNAME"] = auth_username.strip()
    launch_env["URVC_AUTH_PASSWORD"] = auth_password
    print(f"Authentication enabled for user: {auth_username.strip()}")
else:
    launch_env.pop("URVC_AUTH_USERNAME", None)
    launch_env.pop("URVC_AUTH_PASSWORD", None)
    print("WARNING: authentication is disabled; anyone with the share URL can use the app.")

print("Starting CrimsonVC Studio. Open the gradio.live URL shown below.")
print("Stop this cell to shut down the public URL.")
run(
    [uv, "run", "--no-sync", "python", "-m", "ultimate_rvc.web.colab"],
    cwd=WORKSPACE,
    env=launch_env,
)


### Notes and troubleshooting

- If dependency installation leaves the runtime in a broken state, use **Runtime > Disconnect and delete runtime**, reconnect with a GPU, and run all cells again.
- With `google_drive` storage, models, datasets, generated audio, and saved configuration survive runtime deletion. The virtual environment and temporary files intentionally remain on the faster ephemeral disk.
- Google Drive can be slower for training because it handles many small files poorly. For maximum throughput, use `runtime_only` and download the model before deleting the runtime.
- Gradio share links are internet-accessible. Authentication is enabled by default, but the link is intended for temporary personal use rather than permanent hosting.
- Use Quick Train presets as starting points. ContentVec and RMVPE remain compatibility-first defaults; test Spin/Spin-v2 or FCPE only when your own evaluation shows a benefit.